# Lip Forcing 14B
Official: https://github.com/cvlab-kaist/LipForcing . A100 80GB recommended; upstream tested H200 and reports ~50GB peak with text encoder. First download is large. Colab compatibility is experimental.
Run cells in order on a fresh GPU runtime. Use a single visible face and an isolated vocal stem. Input offsets are independent; final output contains test audio, not the full music mix. Not GPU-tested.

In [ ]:
import os, subprocess, sys, shutil, time
from pathlib import Path
from google.colab import drive, files
from IPython.display import Video, display
def run(*args, cwd=None):
    subprocess.run([str(x) for x in args], cwd=cwd, check=True)
run('nvidia-smi')
drive.mount('/content/drive')
root=Path('/content/drive/MyDrive/Alina_LipSync')
root.mkdir(parents=True, exist_ok=True)
run('apt-get','-qq','update')
run('apt-get','-qq','install','-y','ffmpeg')


In [ ]:
repo=Path('/content/LipForcing')
if not repo.exists(): run('git','clone','https://github.com/cvlab-kaist/LipForcing.git',repo)
run(sys.executable,'-m','pip','install','-e',repo)
run(sys.executable,'-m','pip','install','huggingface_hub')
from huggingface_hub import hf_hub_download, snapshot_download
weights=root/'LipForcing_weights'; weights.mkdir(exist_ok=True)
hf_hub_download('JinhyukJang/lipforcing','lipforcing_14b.pth',local_dir=weights)
wan=weights/'Wan2.1-T2V-14B'
for f in ['Wan2.1_VAE.pth','models_t5_umt5-xxl-enc-bf16.pth','google/umt5-xxl/special_tokens_map.json','google/umt5-xxl/spiece.model','google/umt5-xxl/tokenizer.json','google/umt5-xxl/tokenizer_config.json']:
    hf_hub_download('Wan-AI/Wan2.1-T2V-14B',f,local_dir=wan)
snapshot_download('facebook/wav2vec2-base-960h',local_dir=weights/'wav2vec2-base-960h')
for name,url in [('taew2_1.pth','https://raw.githubusercontent.com/madebyollin/taehv/main/taew2_1.pth'),('mask.png','https://raw.githubusercontent.com/bytedance/LatentSync/main/latentsync/utils/mask.png')]:
    if not (weights/name).exists():
        run('curl','-fL','--retry','3','-o',weights/(name+'.part'),url)
        (weights/(name+'.part')).rename(weights/name)
if not (repo/'weights').exists(): (repo/'weights').symlink_to(weights)
run(sys.executable,'scripts/inference/inference_streaming.py','--help',cwd=repo)

In [ ]:
VIDEO_PATH = '' #@param {type:"string"}
AUDIO_PATH = '' #@param {type:"string"}
VIDEO_START = 0.0 #@param {type:"number"}
AUDIO_START = 0.0 #@param {type:"number"}
DURATION = 8.0 #@param {type:"number"}
def choose(path):
    if path:
        p=Path(path).resolve()
    else:
        uploaded=files.upload()
        assert len(uploaded)==1, 'Upload exactly one file'
        p=Path(next(iter(uploaded))).resolve()
    assert p.is_file(), p
    return p
assert DURATION>0 and VIDEO_START>=0 and AUDIO_START>=0
video=choose(VIDEO_PATH)
audio=choose(AUDIO_PATH)
work=Path('/content/alina_inputs'); work.mkdir(exist_ok=True)
v=work/'video.mp4'; a=work/'vocals.wav'
run('ffmpeg','-y','-ss',VIDEO_START,'-i',video,'-t',DURATION,'-an','-vf','fps=25','-c:v','libx264','-pix_fmt','yuv420p',v)
run('ffmpeg','-y','-ss',AUDIO_START,'-i',audio,'-t',DURATION,'-vn','-ac','1','-ar','16000',a)


In [ ]:
out=root/'lipforcing_output.mp4'
PROMPT='a woman singing passionately' #@param {type:"string"}
t=time.perf_counter()
run(sys.executable,'scripts/inference/inference_streaming.py','--ckpt_path',weights/'lipforcing_14b.pth','--vae_path',wan/'Wan2.1_VAE.pth','--wav2vec_path',weights/'wav2vec2-base-960h','--mask_path',weights/'mask.png','--taehv_ckpt',weights/'taew2_1.pth','--text_encoder_path',wan/'models_t5_umt5-xxl-enc-bf16.pth','--prompt',PROMPT,'--video_path',v,'--audio_path',a,'--output_path',out,cwd=repo)
print('Seconds:',time.perf_counter()-t)
assert out.is_file(), out
print('Saved:',out)
display(Video(str(out),embed=True))
# Uncomment to download:
# files.download(str(out))